In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import os

path = kagglehub.dataset_download("hayaalwizrah1/gold-price-prediction-dataset-20002026")
data = pd.read_csv(os.path.join(path, 'gold_data.csv'))

In [2]:
df = data.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()
df.shape

(6468, 24)

In [3]:
df.isnull().sum()

Gold_Open       0
Gold_High       0
Gold_Low        0
Gold_Close      0
Gold_Volume     0
DXY             0
SP500           0
Oil             0
VIX             0
Lag1            0
Lag7            0
MA7             0
MA30            0
EMA20           0
Daily_Return    0
RSI             0
MACD            0
MACD_Signal     0
MACD_Hist       0
BB_Upper        0
BB_Middle       0
BB_Lower        0
ATR             0
Target          0
dtype: int64

In [4]:
# day-of-week / month seasonality, added straight to df so the latest row already has everything it needs later
dow = df.index.dayofweek
df["dow_sin"] = np.sin(2*np.pi*dow/7)
df["dow_cos"] = np.cos(2*np.pi*dow/7)

month = df.index.month
df["month_sin"] = np.sin(2*np.pi*month/12)
df["month_cos"] = np.cos(2*np.pi*month/12)

print(df.shape)

(6468, 28)


In [5]:
df[["Gold_Close","RSI","MACD","DXY","VIX"]].tail()

,Gold_Close,RSI,MACD,DXY,VIX
Date,,,,,
2026-07-24,4067.600098,45.939508,-51.155755,101.470001,18.580000
2026-07-27,4074.500000,46.480858,-46.768670,101.510002,18.670000
2026-07-28,4036.300049,43.862116,-45.845807,101.379997,18.209999
2026-07-29,4034.699951,43.750924,-44.727951,100.800003,20.660000
2026-07-30,4100.100098,49.397363,-38.125311,100.010002,17.090000


In [6]:
df.columns

Index(['Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Close', 'Gold_Volume',
       'DXY', 'SP500', 'Oil', 'VIX', 'Lag1', 'Lag7', 'MA7', 'MA30', 'EMA20',
       'Daily_Return', 'RSI', 'MACD', 'MACD_Signal', 'MACD_Hist', 'BB_Upper',
       'BB_Middle', 'BB_Lower', 'ATR', 'Target', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos'],
      dtype='str')

In [7]:
HORIZON = 21  # ~1 trading month ahead (21 trading days)
df['Target_Direct'] = df["Gold_Close"].shift(-HORIZON)
df = df.dropna(subset=['Target', 'Target_Direct'])

X = df.drop(columns=['Target', 'Target_Direct'])

y_Recursive = df['Target']      # tomorrow's Close
y_Direct = df['Target_Direct']  # Close after 21 days

In [8]:
n = len(X)

train_end = int(n * 0.70)
val_end = int(n * 0.80)   # 70% train + 10% validation

# X split
X_train = X.iloc[:train_end]
X_val   = X.iloc[train_end:val_end]
X_test  = X.iloc[val_end:]

# Recursive target split
y_r_train = y_Recursive.iloc[:train_end]
y_r_val   = y_Recursive.iloc[train_end:val_end]
y_r_test  = y_Recursive.iloc[val_end:]

# Direct target split
y_d_train = y_Direct.iloc[:train_end]
y_d_val   = y_Direct.iloc[train_end:val_end]
y_d_test  = y_Direct.iloc[val_end:]

# Test dates
dates_test = df.index[val_end:]

print("x Train:", X_train.shape)
print("x Validation:", X_val.shape)
print("x Test:", X_test.shape)

print("\nRecursive:")
print("y Train:", y_r_train.shape)
print("y Validation:", y_r_val.shape)
print("y Test:", y_r_test.shape)

print("\nDirect:")
print("y Train:", y_d_train.shape)
print("y Validation:", y_d_val.shape)
print("y Test:", y_d_test.shape)

x Train: (4512, 27)
x Validation: (645, 27)
x Test: (1290, 27)

Recursive:
y Train: (4512,)
y Validation: (645,)
y Test: (1290,)

Direct:
y Train: (4512,)
y Validation: (645,)
y Test: (1290,)


In [9]:
from sklearn.preprocessing import StandardScaler

# 1. Scale X
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_val_s   = scaler_X.transform(X_val)
X_test_s  = scaler_X.transform(X_test)

# 2. Scale Recursive Target
scaler_y_r = StandardScaler()

y_r_train_s = scaler_y_r.fit_transform(y_r_train.values.reshape(-1, 1)).ravel()
y_r_val_s = scaler_y_r.transform(y_r_val.values.reshape(-1, 1)).ravel()
y_r_test_s = scaler_y_r.transform(y_r_test.values.reshape(-1, 1)).ravel()

# 3. Scale Direct Target
scaler_y_d = StandardScaler()

y_d_train_s = scaler_y_d.fit_transform(y_d_train.values.reshape(-1, 1)).ravel()
y_d_val_s = scaler_y_d.transform(y_d_val.values.reshape(-1, 1)).ravel()
y_d_test_s = scaler_y_d.transform(y_d_test.values.reshape(-1, 1)).ravel()

In [10]:
from Deap_learning_f import GoldPricePredictor

predictor = GoldPricePredictor()
predictor.scaler = scaler_X

In [11]:
predictor.train_direct(
    X_train_s,
    y_d_train_s,
    X_val_s,
    y_d_val_s,
    features=X_train.columns.tolist(),
    lr=0.001,
    epochs=200,
    batch_size=32,
    es_patience=10
)

Epoch 1/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3721 - mae: 0.2498 - mse: 0.1222 - val_loss: 0.6223 - val_mae: 0.5061 - val_mse: 0.3090
Epoch 2/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2678 - mae: 0.1573 - mse: 0.0425 - val_loss: 0.5982 - val_mae: 0.4937 - val_mse: 0.2951
Epoch 3/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2412 - mae: 0.1422 - mse: 0.0351 - val_loss: 0.5494 - val_mae: 0.4560 - val_mse: 0.2514
Epoch 4/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2204 - mae: 0.1321 - mse: 0.0303 - val_loss: 0.6298 - val_mae: 0.5465 - val_mse: 0.3547
Epoch 5/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2042 - mae: 0.1255 - mse: 0.0276 - val_loss: 0.6318 - val_mae: 0.5576 - val_mse: 0.3711
Epoch 6/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1887 - mae: 0.1186 - mse: 0.0252 - val_loss: 0.5644 - val_mae: 0.4981 - val_mse: 0.3020
Epoch 7/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1776 - mae: 0.1148 - mse: 0.023

In [12]:
predictor.train_recursive(
    X_train_s,
    y_r_train_s,
    X_val_s,
    y_r_val_s,
    features=X_train.columns.tolist(),
    lr=0.001,
    epochs=200,
    batch_size=32,
    es_patience=10
)

Epoch 1/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3642 - mae: 0.2401 - mse: 0.1184 - val_loss: 0.4066 - val_mae: 0.2884 - val_mse: 0.0999
Epoch 2/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2538 - mae: 0.1416 - mse: 0.0347 - val_loss: 0.3873 - val_mae: 0.2812 - val_mse: 0.0926
Epoch 3/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2202 - mae: 0.1198 - mse: 0.0252 - val_loss: 0.3693 - val_mae: 0.2744 - val_mse: 0.0866
Epoch 4/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1923 - mae: 0.1029 - mse: 0.0185 - val_loss: 0.4049 - val_mae: 0.3207 - val_mse: 0.1207
Epoch 5/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1735 - mae: 0.0941 - mse: 0.0159 - val_loss: 0.4423 - val_mae: 0.3674 - val_mse: 0.1548
Epoch 6/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1605 - mae: 0.0899 - mse: 0.0147 - val_loss: 0.4204 - val_mae: 0.3541 - val_mse: 0.1458
Epoch 7/200
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1456 - mae: 0.0831 - mse: 0.012

In [13]:
direct_pred_s = predictor.direct_model.predict(X_test_s, verbose=0).ravel()
direct_pred = scaler_y_d.inverse_transform(direct_pred_s.reshape(-1, 1)).ravel()

recursive_pred_s = predictor.recursive_model.predict(X_test_s, verbose=0 ).ravel()
recursive_pred = scaler_y_r.inverse_transform(recursive_pred_s.reshape(-1, 1)).ravel()

In [14]:
direct_metrics = predictor.evaluate(
    predictor.direct_model,
    X_test_s,
    y_d_test_s,
    label="Direct"
)

RMSE : 1.7908365652468623
MAE  : 1.5228279254401595
R²   : 0.30396595676817606


In [18]:
direct_metrics = predictor.evaluate(
    predictor.recursive_model,
    X_test_s,
    y_d_test_s,
    label="Recursive"
)

RMSE : 0.9750459426150359
MAE  : 0.7989194902507127
R²   : 0.793666980343007
